In [ ]:
# rogii-ultranote-v4 — UltraNote v3 (Roman+XGB+NM) + Universal Kriging surface
# Sprint 1 winner: UK per-surface (weighted RMSE 57 vs plane-fit 119)
import subprocess, sys, os
for pkg in ["numba"]:
    if subprocess.run([sys.executable,"-m","pip","show",pkg],capture_output=True).returncode!=0:
        subprocess.run([sys.executable,"-m","pip","install",pkg,"--quiet"])
os.environ["NUMBA_CACHE_DIR"]="/kaggle/working/.numba"
os.makedirs("/kaggle/working/.numba",exist_ok=True)
print("setup ok")


In [ ]:
# Inlined surface module
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial import cKDTree
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.linear_model import HuberRegressor  # for robust quadratic
FORMATIONS=['ANCC','ASTNU','ASTNL','EGFDU','EGFDL','BUDA']
STRIKE_DEG=48.0
KM_TO_FT=3280.84

def rotate_to_strike(xy, strike_deg=STRIKE_DEG):
    theta=-np.radians(strike_deg)
    R=np.array([[np.cos(theta),-np.sin(theta)],[np.sin(theta),np.cos(theta)]])
    return xy @ R.T

# Inlined load_horizontal for FormationSurfaceV2's centroid building
from glob import glob
def _find_data():
    for p in [Path('/kaggle/input/rogii-wellbore-geology-prediction'),
              Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction')]:
        if (p/'train').exists(): return p
    raise FileNotFoundError('Data not found')
_DATA_DIR = _find_data()
def load_horizontal(well_id, split='train'):
    p = _DATA_DIR / split / f'{well_id}__horizontal_well.csv'
    return pd.read_csv(p)
"""Base interface for surface methods.

Every method must implement:
  fit(centroids, surface_values)
  predict(query_xy) -> (mean, std)

`centroids`: shape (n, 2) array of well (X, Y) medians.
`surface_values`: shape (n,) array of surface Z (or thickness).
`query_xy`: shape (q, 2) array of query points.
Returns mean (q,) and std (q,) — std may be all-zero if method has no uncertainty.
"""
from abc import ABC, abstractmethod
import numpy as np


class SurfaceMethod(ABC):
    """Abstract base for one-surface predictors."""

    name: str = "base"

    def __init__(self, **params):
        self.params = params
        self._fitted = False

    @abstractmethod
    def fit(self, centroids: np.ndarray, values: np.ndarray) -> "SurfaceMethod":
        ...

    @abstractmethod
    def predict(self, query_xy: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """Returns (mean, std). std may be all-zero if no uncertainty."""
        ...

    def is_fitted(self) -> bool:
        return self._fitted

"""Baseline #1: konbu plane-fit-KNN.

For each query (X,Y):
  1. Find K nearest training centroids in Euclidean (X,Y)
  2. Inverse-distance weights w_i = 1/(d_i + eps)
  3. Solve weighted least-squares plane Z = a + bX + cY on K neighbors
  4. Evaluate at query
"""
import numpy as np
from scipy.spatial import cKDTree



class PlaneFitKNN(SurfaceMethod):
    name = "plane_fit_knn"

    def __init__(self, k: int = 10):
        super().__init__(k=k)
        self.k = k

    def fit(self, centroids: np.ndarray, values: np.ndarray) -> "PlaneFitKNN":
        self.centroids = np.asarray(centroids, np.float64)
        self.values = np.asarray(values, np.float64)
        # Standardize for KDTree distance
        self.scale = np.where(self.centroids.std(0) < 1e-3, 1.0, self.centroids.std(0))
        self.tree = cKDTree(self.centroids / self.scale)
        self._fitted = True
        return self

    def predict(self, query_xy: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        q = np.atleast_2d(np.asarray(query_xy, np.float64))
        nf = min(self.k + 5, len(self.centroids))
        dist, idx = self.tree.query(q / self.scale, k=nf, workers=-1)
        # Take K closest (no exclusion — caller should not pass query == train centroid)
        ord = np.argpartition(dist, min(self.k - 1, nf - 1), axis=1)[:, : self.k]
        dk = np.take_along_axis(dist, ord, axis=1)
        ik = np.take_along_axis(idx, ord, axis=1)
        valid = np.isfinite(dk)
        w = np.where(valid, 1.0 / (dk + 1e-3), 0.0)

        xn = self.centroids[ik, 0]
        yn = self.centroids[ik, 1]
        fn = self.values[ik]
        wx = w * xn
        wy = w * yn

        # Solve 3x3 linear system per query: [a + b*X + c*Y]
        n_q = len(q)
        A = np.zeros((n_q, 3, 3))
        A[:, 0, 0] = (wx * xn).sum(1)
        A[:, 0, 1] = (wx * yn).sum(1)
        A[:, 0, 2] = wx.sum(1)
        A[:, 1, 0] = A[:, 0, 1]
        A[:, 1, 1] = (wy * yn).sum(1)
        A[:, 1, 2] = wy.sum(1)
        A[:, 2, 0] = A[:, 0, 2]
        A[:, 2, 1] = A[:, 1, 2]
        A[:, 2, 2] = w.sum(1)

        # Regularize
        for i in range(3):
            A[:, i, i] += 1e-9

        rhs = np.column_stack([
            (wx * fn).sum(1),
            (wy * fn).sum(1),
            (w * fn).sum(1),
        ])

        try:
            coef = np.linalg.solve(A, rhs[:, :, None]).squeeze(-1)
        except np.linalg.LinAlgError:
            coef = np.zeros((n_q, 3))
            for r in range(n_q):
                try:
                    coef[r] = np.linalg.pinv(A[r]) @ rhs[r]
                except Exception:
                    pass

        pred = (q[:, 0] * coef[:, 0] + q[:, 1] * coef[:, 1] + coef[:, 2])
        # No native uncertainty; return zeros
        std = np.zeros_like(pred)
        return pred.astype(np.float64), std

"""Method #3: Anisotropic Matérn-3/2 GP with ARD constraints.

GP is fit in rotated (strike, dip) coordinate system:
  - Major axis along strike (≈48° NE-SW from data)
  - Minor axis down dip (≈138° ESE)

Lengthscales:
  - Strike init = 12 km (40 000 ft), bounds [3 km, 50 km]
  - Dip init = 4 km (13 000 ft), bounds [1 km, 20 km]
  - Constraint l_strike >= l_dip (geologic prior)

Output uncertainty (sigma) is genuine GP posterior std.

Fit via L-BFGS-B with multiple random restarts to avoid local optima.
"""
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel, Matern, WhiteKernel,
)



# Defaults from data measurement
STRIKE_DEG = 48.0  # axis along strike (math convention; compass equivalent ~42°)
DIP_DEG = 138.0    # along dip
# In feet: 1 km = 3280.84 ft
KM_TO_FT = 3280.84


def rotate_to_strike(xy: np.ndarray, strike_deg: float = STRIKE_DEG) -> np.ndarray:
    """Rotate (X,Y) so first axis is along strike, second along dip."""
    # Math convention: rotate by -strike to align strike with X-axis
    theta = -np.radians(strike_deg)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    R = np.array([[cos_t, -sin_t], [sin_t, cos_t]])
    return xy @ R.T


class AnisotropicMaternGP(SurfaceMethod):
    name = "gp_matern"

    def __init__(
        self,
        strike_deg: float = STRIKE_DEG,
        l_strike_init_ft: float = 15 * KM_TO_FT,    # 15 km along strike
        l_dip_init_ft: float = 5 * KM_TO_FT,         # 5 km down dip
        l_strike_bounds_ft: tuple[float, float] = (3 * KM_TO_FT, 100 * KM_TO_FT),
        l_dip_bounds_ft: tuple[float, float] = (1 * KM_TO_FT, 50 * KM_TO_FT),
        nu: float = 1.5,                             # Matérn 3/2
        n_restarts: int = 5,
    ):
        super().__init__(
            strike_deg=strike_deg,
            l_strike_init_ft=l_strike_init_ft,
            l_dip_init_ft=l_dip_init_ft,
        )
        self.strike_deg = strike_deg
        self.l_strike_init_ft = l_strike_init_ft
        self.l_dip_init_ft = l_dip_init_ft
        self.l_strike_bounds_ft = l_strike_bounds_ft
        self.l_dip_bounds_ft = l_dip_bounds_ft
        self.nu = nu
        self.n_restarts = n_restarts

    def fit(self, centroids: np.ndarray, values: np.ndarray) -> "AnisotropicMaternGP":
        xy = np.asarray(centroids, np.float64)
        z = np.asarray(values, np.float64)
        # Center for numerical stability
        self.x_center = xy.mean(0)
        xy_c = xy - self.x_center
        self.z_mean = float(z.mean())
        z_c = z - self.z_mean

        # Rotate to strike-dip frame
        xy_rot = rotate_to_strike(xy_c, self.strike_deg)

        # Build anisotropic Matérn kernel
        # ARD with 2 lengthscales (strike + dip)
        # ConstantKernel scales output variance
        # WhiteKernel = noise + nugget
        z_var = float(np.var(z_c)) + 1.0
        kernel = (
            ConstantKernel(constant_value=z_var, constant_value_bounds=(z_var * 0.01, z_var * 100))
            * Matern(
                length_scale=[self.l_strike_init_ft, self.l_dip_init_ft],
                length_scale_bounds=[self.l_strike_bounds_ft, self.l_dip_bounds_ft],
                nu=self.nu,
            )
            + WhiteKernel(noise_level=z_var * 0.01,
                          noise_level_bounds=(z_var * 0.001, z_var * 1.0))
        )

        self.gp = GaussianProcessRegressor(
            kernel=kernel,
            n_restarts_optimizer=self.n_restarts,
            normalize_y=False,  # we handle centering manually
            alpha=1e-6,
            random_state=42,
        )
        self.gp.fit(xy_rot, z_c)
        self._fitted = True
        return self

    def predict(self, query_xy: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        q = np.atleast_2d(np.asarray(query_xy, np.float64))
        q_c = q - self.x_center
        q_rot = rotate_to_strike(q_c, self.strike_deg)
        mean, std = self.gp.predict(q_rot, return_std=True)
        return mean + self.z_mean, std

    def kernel_summary(self) -> str:
        if not self._fitted:
            return "(not fitted)"
        return str(self.gp.kernel_)

"""Method #6: Universal kriging = global trend + GP residual.

Trend: best linear plane Z = a + bX + cY (regional dip)
Residual: GP с anisotropic Matérn-3/2 fit on (Z - trend_pred)

Total prediction: trend_at_query + gp_residual_at_query.
GP std comes from residual GP (trend itself is deterministic).
"""
import numpy as np



class UniversalKriging(SurfaceMethod):
    name = "universal_kriging"

    def __init__(self, n_restarts: int = 1, **gp_kwargs):
        super().__init__(n_restarts=n_restarts, **gp_kwargs)
        gp_kwargs["n_restarts"] = n_restarts
        self.gp_kwargs = gp_kwargs

    def fit(self, centroids: np.ndarray, values: np.ndarray) -> "UniversalKriging":
        xy = np.asarray(centroids, np.float64)
        z = np.asarray(values, np.float64)
        # Fit linear trend Z = a + bX + cY
        A = np.column_stack([xy, np.ones(len(xy))])
        coef, *_ = np.linalg.lstsq(A, z, rcond=None)
        self.trend_coef = coef
        z_resid = z - (A @ coef)
        # GP on residual
        self.gp = AnisotropicMaternGP(**self.gp_kwargs).fit(xy, z_resid)
        self._fitted = True
        return self

    def predict(self, query_xy: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        q = np.atleast_2d(np.asarray(query_xy, np.float64))
        # Trend
        A = np.column_stack([q, np.ones(len(q))])
        trend_pred = A @ self.trend_coef
        # GP residual
        resid_mean, resid_std = self.gp.predict(q)
        return trend_pred + resid_mean, resid_std

"""Method #4: Hierarchical surface model.

Architecture:
  D_ANCC(X,Y)        - structural datum (positive depth = -Z)
  h1, h2, ..., h5    - positive thickness fields between consecutive tops

Reconstruct any top:
  D_ASTNU = D_ANCC + h1
  D_ASTNL = D_ASTNU + h2
  D_EGFDU = D_ASTNL + h3
  D_EGFDL = D_EGFDU + h4
  D_BUDA  = D_EGFDL + h5
  ANCC    = -D_ANCC, etc.

Positivity of h_i enforced via log-transform: predict log(h_i), then exp.

The orchestrator is a SurfaceMethod-like wrapper but predicts ALL 6 tops at
once. Underlying method (e.g. AnisotropicMaternGP) is reused per field.
"""
import numpy as np



class HierarchicalSurface:
    """Predicts all 6 formation tops jointly, ensuring stratigraphic order.

    Train inputs:
      centroids: (n, 2) X,Y per well
      tops_dict: {ANCC, ASTNU, ASTNL, EGFDU, EGFDL, BUDA -> (n,) Z values (negative subsea)}
    """

    def __init__(self, base_method_factory):
        """base_method_factory: callable that returns a fresh SurfaceMethod instance.
        e.g. lambda: AnisotropicMaternGP()
        """
        self.base_factory = base_method_factory
        self.models = {}

    def _to_depth(self, z: np.ndarray) -> np.ndarray:
        return -z  # positive depth

    def _from_depth(self, d: np.ndarray) -> np.ndarray:
        return -d

    def fit(self, centroids: np.ndarray, tops_dict: dict[str, np.ndarray]) -> "HierarchicalSurface":
        # Compute D_ANCC and 5 thicknesses
        D = {}
        for f in FORMATIONS:
            D[f] = self._to_depth(np.asarray(tops_dict[f], np.float64))

        h = {
            "h1": D["ASTNU"] - D["ANCC"],
            "h2": D["ASTNL"] - D["ASTNU"],
            "h3": D["EGFDU"] - D["ASTNL"],
            "h4": D["EGFDL"] - D["EGFDU"],
            "h5": D["BUDA"] - D["EGFDL"],
        }
        # Sanity: all h_i must be >= 0 (we verified 100% on train)
        for k, v in h.items():
            if (v < -0.5).any():  # tolerance for numerical noise
                bad = (v < -0.5).sum()
                print(f"WARN: {bad} wells have negative {k} (>{v[v<-0.5].min():.1f} ft); clipping to small positive")
            h[k] = np.maximum(v, 1e-3)  # avoid log(0)

        # Fit datum on D_ANCC
        self.models["D_ANCC"] = self.base_factory().fit(centroids, D["ANCC"])
        # Fit thicknesses on log(h_i)
        for k, v in h.items():
            self.models[k] = self.base_factory().fit(centroids, np.log(v))

        self._fitted = True
        return self

    def predict_all(self, query_xy: np.ndarray) -> dict[str, tuple[np.ndarray, np.ndarray]]:
        """Returns {formation_name: (mean_Z, std_Z)} for all 6 formations."""
        q = np.atleast_2d(np.asarray(query_xy, np.float64))

        # Predict datum
        d_ancc_mean, d_ancc_std = self.models["D_ANCC"].predict(q)

        # Predict thicknesses (in log-space) and exp back
        h_mean = {}
        h_std = {}
        for k in ["h1", "h2", "h3", "h4", "h5"]:
            log_mean, log_std = self.models[k].predict(q)
            # Lognormal mean/std: E[exp(X)] = exp(μ+σ²/2); Var = exp(2μ+σ²)(exp(σ²)-1)
            h_mean[k] = np.exp(log_mean + 0.5 * log_std ** 2)
            h_std[k] = h_mean[k] * np.sqrt(np.exp(log_std ** 2) - 1.0)
            # Ensure positivity numerically
            h_mean[k] = np.maximum(h_mean[k], 1e-3)
            h_std[k] = np.minimum(h_std[k], h_mean[k])  # cap

        # Reconstruct all 6 surfaces
        # D_ANCC, D_ASTNU = D_ANCC + h1, ...
        D_means = {"ANCC": d_ancc_mean}
        D_stds = {"ANCC": d_ancc_std}
        D_means["ASTNU"] = D_means["ANCC"] + h_mean["h1"]
        D_stds["ASTNU"] = np.sqrt(D_stds["ANCC"] ** 2 + h_std["h1"] ** 2)
        D_means["ASTNL"] = D_means["ASTNU"] + h_mean["h2"]
        D_stds["ASTNL"] = np.sqrt(D_stds["ASTNU"] ** 2 + h_std["h2"] ** 2)
        D_means["EGFDU"] = D_means["ASTNL"] + h_mean["h3"]
        D_stds["EGFDU"] = np.sqrt(D_stds["ASTNL"] ** 2 + h_std["h3"] ** 2)
        D_means["EGFDL"] = D_means["EGFDU"] + h_mean["h4"]
        D_stds["EGFDL"] = np.sqrt(D_stds["EGFDU"] ** 2 + h_std["h4"] ** 2)
        D_means["BUDA"] = D_means["EGFDL"] + h_mean["h5"]
        D_stds["BUDA"] = np.sqrt(D_stds["EGFDL"] ** 2 + h_std["h5"] ** 2)

        # Convert back to subsea Z
        out = {}
        for f in FORMATIONS:
            mean_z = self._from_depth(D_means[f])
            out[f] = (mean_z, D_stds[f])
        return out

    def is_fitted(self):
        return getattr(self, "_fitted", False)

"""Method #7: Edge-aware UK + KNN hybrid (V2 — static density).

Motivation: Universal Kriging wins in dense interior regions but degrades at the
boundary of training support, where it must extrapolate. Plane-fit-KNN is purely
local and degrades less abruptly there.

Strategy: density is computed *once* using positions of all known wells (training
+ test, since well positions are KNOWN at test time and not target-leaking). For
each query point we use that precomputed static density to decide a blend weight:
    alpha = sigmoid((density - threshold) / width)
    pred  = alpha * UK + (1 - alpha) * KNN

This is the V2 protocol that beat UK alone by 31.6 ft on edge CV and 24.3 ft on
spatial CV (ANCC OOF) — see experiments/comparison_maps/. The earlier V1 (which
recomputed density per fold) failed on spatial CV because CV-induced sparsity
triggered fallback even for interior queries.

Use:
    m = EdgeAwareHybrid().fit(train_xy, train_z, all_xy=full_dataset_xy)
    mean, std = m.predict(query_xy)

If `all_xy` is not provided, falls back to V1 behaviour (density from fit-time
training centroids only).
"""
import numpy as np
from scipy.spatial import cKDTree



class EdgeAwareHybrid(SurfaceMethod):
    name = "edge_aware_hybrid"

    def __init__(
        self,
        density_radius: float = 0.3,        # in standardized-bbox units
        density_threshold_pct: float = 10,  # auto-set threshold to this density percentile
        knn_k: int = 10,
        uk_n_restarts: int = 2,
        **uk_kwargs,
    ):
        super().__init__(
            density_radius=density_radius,
            density_threshold_pct=density_threshold_pct,
            knn_k=knn_k,
            uk_n_restarts=uk_n_restarts,
        )
        self.density_radius = density_radius
        self.density_threshold_pct = density_threshold_pct
        self.knn_k = knn_k
        self.uk_n_restarts = uk_n_restarts
        self.uk_kwargs = uk_kwargs

    def fit(self, centroids: np.ndarray, values: np.ndarray,
            all_xy: np.ndarray | None = None) -> "EdgeAwareHybrid":
        """Fit. `all_xy` is the full set of well positions (train+test) used for
        the static density calculation. If None, density uses `centroids` only
        (V1 behaviour — pessimistic in CV).
        """
        xy = np.asarray(centroids, np.float64)
        z = np.asarray(values, np.float64)

        # Standardise for the density KDTree using positions from all_xy if given
        ref_xy = np.asarray(all_xy, np.float64) if all_xy is not None else xy
        self.scale = np.where(ref_xy.std(0) < 1e-3, 1.0, ref_xy.std(0))
        self.density_tree = cKDTree(ref_xy / self.scale)

        # Auto-set threshold from density distribution at ref points
        nb_self = self.density_tree.query_ball_point(ref_xy / self.scale, r=self.density_radius)
        density_self = np.array([len(n) - 1 for n in nb_self])
        self.density_threshold = float(np.percentile(density_self, self.density_threshold_pct))
        p25 = float(np.percentile(density_self, 25))
        self.density_width = max(2.0, p25 - self.density_threshold)

        # Fit both methods on training data
        self.uk = UniversalKriging(n_restarts=self.uk_n_restarts, **self.uk_kwargs).fit(xy, z)
        self.knn = PlaneFitKNN(k=self.knn_k).fit(xy, z)
        self._fitted = True
        return self

    def predict(self, query_xy: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        q = np.atleast_2d(np.asarray(query_xy, np.float64))

        # Density: number of static-set centroids within `density_radius` of each query
        nb_idx = self.density_tree.query_ball_point(q / self.scale, r=self.density_radius)
        density = np.fromiter((len(n) for n in nb_idx), dtype=np.float64, count=len(q))

        z_score = (density - self.density_threshold) / max(self.density_width, 1e-6)
        alpha = 1.0 / (1.0 + np.exp(-z_score))

        uk_mean, uk_std = self.uk.predict(q)
        knn_mean, knn_std = self.knn.predict(q)

        mean = alpha * uk_mean + (1.0 - alpha) * knn_mean
        std = alpha * uk_std + (1.0 - alpha) * knn_std
        std = std + (1 - 4 * alpha * (1 - alpha)) * np.abs(uk_mean - knn_mean) * 0.1
        return mean, std

    def predict_with_alpha(self, query_xy: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        q = np.atleast_2d(np.asarray(query_xy, np.float64))
        nb_idx = self.density_tree.query_ball_point(q / self.scale, r=self.density_radius)
        density = np.fromiter((len(n) for n in nb_idx), dtype=np.float64, count=len(q))
        z_score = (density - self.density_threshold) / max(self.density_width, 1e-6)
        alpha = 1.0 / (1.0 + np.exp(-z_score))
        uk_mean, _ = self.uk.predict(q)
        knn_mean, _ = self.knn.predict(q)
        mean = alpha * uk_mean + (1.0 - alpha) * knn_mean
        return mean, alpha, density

"""Anchor v2 — drop-in replacement for src/anchor.py FormationPlaneKNN.

Uses Sprint 1 surface lab winner (or any specified method).

Same interface as FormationPlaneKNN:
  - __init__(train_well_ids, k=10)
  - impute(xy_query, exclude_well=None) -> (preds (n_q, 6), nearest_dist)

But internally uses HierarchicalSurface with configurable base method (GP by default).
"""
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree



def _build_centroids(train_well_ids: list[str]) -> tuple[pd.DataFrame, np.ndarray, dict[str, np.ndarray]]:
    rows = []
    for w in train_well_ids:
        try:
            hw = load_horizontal(w, "train")
        except FileNotFoundError:
            continue
        if any(c not in hw.columns for c in FORMATIONS):
            continue
        df = hw[["X", "Y"] + FORMATIONS].dropna()
        if len(df) == 0:
            continue
        row = {
            "wid": w,
            "x": float(df["X"].median()),
            "y": float(df["Y"].median()),
        }
        for c in FORMATIONS:
            row[c] = float(df[c].median())
        rows.append(row)
    df = pd.DataFrame(rows)
    centroids = df[["x", "y"]].to_numpy()
    tops = {f: df[f].to_numpy() for f in FORMATIONS}
    return df, centroids, tops


class FormationSurfaceV2:
    """Drop-in replacement for FormationPlaneKNN with hierarchical GP backend."""

    def __init__(
        self,
        train_well_ids: list[str],
        method: str = "gp_hierarchical",
        k: int = 10,
    ):
        self.method = method
        self.k = k
        self.df, self.centroids, self.tops = _build_centroids(train_well_ids)
        self.wmap = {w: i for i, w in enumerate(self.df["wid"])}
        # Standardize for fast KDTree fallback
        self.scale = np.where(self.centroids.std(0) < 1e-3, 1.0, self.centroids.std(0))
        self.tree = cKDTree(self.centroids / self.scale)

        # Build the surface model
        if method == "plane_knn":
            self.surface = PlaneFitKNN(k=k)
            self._fit_per_surface()
        elif method == "gp_per_surface":
            self.surface = None
            self._fit_per_surface_gp()
        elif method == "uk_per_surface":
            self.surface = None
            self._fit_per_surface_uk()
        elif method == "gp_hierarchical":
            self.surface = HierarchicalSurface(base_method_factory=lambda: AnisotropicMaternGP(n_restarts=3))
            self.surface.fit(self.centroids, self.tops)
        elif method == "uk_hierarchical":
            self.surface = HierarchicalSurface(base_method_factory=lambda: UniversalKriging(n_restarts=3))
            self.surface.fit(self.centroids, self.tops)
        elif method == "edge_aware_hybrid":
            self.surface = None
            self._fit_per_surface_edge_aware()
        else:
            raise ValueError(f"unknown method {method}")

    def _fit_per_surface(self):
        self.per_surface = {f: PlaneFitKNN(k=self.k).fit(self.centroids, self.tops[f]) for f in FORMATIONS}

    def _fit_per_surface_gp(self):
        self.per_surface = {f: AnisotropicMaternGP(n_restarts=3).fit(self.centroids, self.tops[f]) for f in FORMATIONS}

    def _fit_per_surface_uk(self):
        self.per_surface = {f: UniversalKriging(n_restarts=3).fit(self.centroids, self.tops[f]) for f in FORMATIONS}

    def _fit_per_surface_edge_aware(self):
        # static density uses ALL training centroids (positions only, no leak)
        self.per_surface = {
            f: EdgeAwareHybrid(uk_n_restarts=2).fit(self.centroids, self.tops[f], all_xy=self.centroids)
            for f in FORMATIONS
        }

    def impute(self, xy_query: np.ndarray, exclude_well: str | None = None,
               self_wid: str | None = None, k: int | None = None) -> tuple[np.ndarray, np.ndarray]:
        """Returns (preds (n_q, 6), nearest distances).

        Accepts both `exclude_well` (V2 style) and `self_wid` (konbu-roman style)
        for compatibility. For GP/UK methods, exclude is ignored (full-train fit
        is used). `k` ignored — surface methods are global, not KNN.
        """
        if exclude_well is None and self_wid is not None:
            exclude_well = self_wid
        q = np.atleast_2d(xy_query)
        if self.method in ("plane_knn", "gp_per_surface", "uk_per_surface", "edge_aware_hybrid"):
            preds = np.zeros((len(q), 6))
            for i, f in enumerate(FORMATIONS):
                m = self.per_surface[f]
                preds[:, i], _ = m.predict(q)
        elif self.method in ("gp_hierarchical", "uk_hierarchical"):
            out = self.surface.predict_all(q)
            preds = np.column_stack([out[f][0] for f in FORMATIONS])
        else:
            raise ValueError(self.method)

        # Distance to nearest centroid (in standardized units)
        dist, _ = self.tree.query(q / self.scale, k=1, workers=-1)
        return preds, dist


In [ ]:
#!/usr/bin/env python3
# Auto-extracted from Roman tamrazov/rogii-super-solution-lb-top-3

# === md cell 0 ===
# # ROGII — Sub-9 Solution v2 (Tuned LGB+CB)
# **From 10.184 → target < 9.0** | ~55 min on T4×2
# 
# ### Strategy: polish the proven pipeline, don't add noise
# - LGB: num_leaves=255, lr=0.025, 8000 iter + better regularisation
# - CB: depth=7, 8000 iter, lr=0.025, border_count=254 (GPU quality)
# - Numba JIT beam ±2 delta, 7 configs
# - Multi-scale NCC self-corr (hw=8,15,25) → 3 TVT signals
# - WLS b_well (recent rows upweighted, decay=0.02)
# - Formation consensus std/range → uncertainty features
# - Per-formation known-zone RMSE → which surface to trust
# - Inter-signal std → master uncertainty feature
# - GR envelope + energy (rolling max / RMS)
# - Richer tw_diff: 4th family anchored at PF-ANCC
# - Monotonic GR trend (linear detrended residual)
# - Smart ridge stack: 4 models, positive=True

# === cell 1 ===
import subprocess, sys, os
for pkg in ["numba"]:
    if subprocess.run([sys.executable,"-m","pip","show",pkg],capture_output=True).returncode!=0:
        subprocess.run([sys.executable,"-m","pip","install",pkg,"--quiet"])
_kaggle_working = "/kaggle/working" if os.path.isdir("/kaggle/working") else None
_default_numba = (_kaggle_working + "/.numba") if _kaggle_working else "./.numba"
os.environ["NUMBA_CACHE_DIR"]=os.environ.get("NUMBA_CACHE_DIR",_default_numba)
os.makedirs(os.environ["NUMBA_CACHE_DIR"],exist_ok=True)
print("setup ok")


# === cell 2 ===
from pathlib import Path
from scipy.interpolate import interp1d
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from catboost import CatBoostRegressor, Pool
import xgboost as xgb
from numba import njit
from scipy.optimize import minimize
from joblib import Parallel, delayed
import lightgbm as lgb
import numpy as np, pandas as pd
import gc, time, multiprocessing, warnings
warnings.filterwarnings("ignore")

SEED=42; np.random.seed(SEED)
# v2: bump threads — beleriand has 80 cores, Kaggle has 4. NM blend benefits from more.
NCPU=int(os.environ.get("ROGII_NCPU", min(16, multiprocessing.cpu_count())))

def _find():
    import os
    env = os.environ.get('ROGII_DATA_DIR')
    if env:
        p = Path(env)
        if (p/'train').exists() and (p/'sample_submission.csv').exists():
            return p
    for p in [Path('/home/manushinma/kaggle/rogii/rogii-wellbore-geology-prediction/data'),
              Path('/home/max/kaggle/rogii/rogii-wellbore-geology-prediction/data'),
              Path('/kaggle/input/rogii-wellbore-geology-prediction'),
              Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction')]:
        if (p/'train').exists() and (p/'sample_submission.csv').exists():
            return p
    raise FileNotFoundError('Data not found in any expected location')

DATA=_find(); TRAIN_DIR=DATA/"train"; TEST_DIR=DATA/"test"
SAMPLE=DATA/"sample_submission.csv"
_default_out = "/kaggle/working/submission.csv" if _kaggle_working else "./submission.csv"
OUT=Path(os.environ.get("ROGII_OUT",_default_out))

FORMATIONS=["ANCC","ASTNU","ASTNL","EGFDU","EGFDL","BUDA"]
PLANE_K=10; DENSE_SPW=60; DENSE_K=20; N_SPLITS=5

# 7 beam configs: diverse move_cost/emit_scale/smooth_radius
BEAMS=[
    (10,20.0,144.0,2,"cons"),
    (10, 8.0, 64.0,2,"loose"),
    ( 8,35.0,220.0,1,"vcons"),
    (10,14.0, 90.0,5,"sm5"),
    (20, 4.0, 36.0,3,"vloose"),
    (12,12.0,100.0,3,"mid"),
    (15,25.0,180.0,2,"stiff"),
]

# PF params (N=500, Numba-affordable)
PF_N=500; ANCC_N=500
PF_MOM=0.993; PF_VN=0.005; PF_PN=0.01
PF_GR_SIG_MIN=10.; PF_GR_SIG_MAX=60.; PF_GR_SIG_DEF=30.
PF_INIT_V_STD=0.02; PF_INIT_SPR=0.5; PF_RESAMP=0.5
PF_ROUGH_P=0.2; PF_ROUGH_V=0.003; PF_GR_WIN=5; PF_GR_WT=0.3
ANCC_ALPHA=0.998; ANCC_RN=0.002; ANCC_PN=0.005
ANCC_IR=0.01; ANCC_IS=0.3; ANCC_RP=0.1; ANCC_RR=0.001

# ── Model hyperparams (tuned, no XGBoost) ───────────────────────
LGB_BASE=dict(
    boosting_type="gbdt",
    num_leaves=255,          # was 127
    min_child_samples=15,    # was 20 — allows finer splits
    subsample=0.75,
    subsample_freq=1,
    colsample_bytree=0.75,
    reg_lambda=3.0,          # reduced from 5
    reg_alpha=0.05,
    min_split_gain=0.01,
    objective="regression",
    verbose=-1, n_jobs=-1,
    device_type="gpu", gpu_use_dp=False, max_bin=255,
)
# Three seeds, each with slightly different lr → diversity
LGB_CONFIGS=[
    dict(learning_rate=0.025, n_estimators=8000, seed=42),
    dict(learning_rate=0.020, n_estimators=8000, seed=7),
    dict(learning_rate=0.030, n_estimators=8000, seed=123),
]

CB_PARAMS=dict(
    iterations=8000,
    learning_rate=0.025,
    depth=7,
    l2_leaf_reg=2.0,
    min_data_in_leaf=15,
    bootstrap_type='Bernoulli',  # required for subsample (newer CatBoost default is bayesian)
    subsample=0.75,
    border_count=254,
    loss_function="RMSE",
    random_seed=42,
    task_type="GPU",
    devices=os.environ.get("ROGII_CB_DEVICES", "0"),  # single-GPU default; multi-GPU users set explicitly
    od_type="Iter",
    od_wait=300,
    verbose=0,
)

# v2: XGBoost added (Karnakbaev's diversity)
XGB_PARAMS=dict(
    n_estimators=2500,
    learning_rate=0.025,
    max_depth=8,
    min_child_weight=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2.0,
    reg_alpha=0.05,
    objective='reg:squarederror',
    tree_method='hist',
    device='cuda',
    eval_metric='rmse',
    early_stopping_rounds=200,
    verbosity=0,
)

import subprocess as _s
print("GPUs:",_s.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
      capture_output=True,text=True).stdout.strip())
print(f"CPUs={NCPU} | wells={len(list(TRAIN_DIR.glob('*__horizontal_well.csv')))}")


# === cell 3 ===
# ── Numba JIT Beam (±2 delta, cached) ────────────────────────────
@njit(cache=True)
def _beam_jit(sgr,tw_gr,si,BS,mc,es):
    n=len(sgr); nt=len(tw_gr); MAX=BS*5
    bidx=np.zeros(BS,np.int64); bidx[0]=si
    bcost=np.full(BS,1e30);     bcost[0]=0.; bn=np.int64(1)
    hI=np.zeros((n,BS),np.int64); hP=np.zeros((n,BS),np.int64)
    cI=np.zeros(MAX,np.int64); cC=np.full(MAX,1e30); cP=np.zeros(MAX,np.int64)
    for step in range(n):
        gv=sgr[step]; nc=np.int64(0)
        for bi in range(bn):
            idx=bidx[bi]; cost=bcost[bi]
            for d in range(-2,3):
                ni=idx+d
                if ni<0 or ni>=nt: continue
                tot=cost+(gv-tw_gr[ni])**2/es+mc*(d if d>=0 else -d)
                fnd=np.int64(-1)
                for ci in range(nc):
                    if cI[ci]==ni: fnd=ci; break
                if fnd>=0:
                    if tot<cC[fnd]: cC[fnd]=tot; cP[fnd]=bi
                else:
                    if nc<MAX: cI[nc]=ni; cC[nc]=tot; cP[nc]=bi; nc+=1
        kept=min(BS,nc)
        for i in range(kept):
            mi=i
            for j in range(i+1,nc):
                if cC[j]<cC[mi]: mi=j
            if mi!=i:
                cI[i],cI[mi]=cI[mi],cI[i]; cC[i],cC[mi]=cC[mi],cC[i]; cP[i],cP[mi]=cP[mi],cP[i]
        hI[step,:kept]=cI[:kept]; hP[step,:kept]=cP[:kept]
        bidx[:kept]=cI[:kept]; bcost[:kept]=cC[:kept]; bn=kept
    best=np.int64(0)
    for b in range(1,bn):
        if bcost[b]<bcost[best]: best=b
    path=np.zeros(n,np.int64); b=best
    for s in range(n-1,-1,-1): path[s]=hI[s,b]; b=hP[s,b]
    return path

def _nn(a,v):
    i=int(np.searchsorted(a,v,'left'))
    if i>=len(a): return len(a)-1
    if i>0 and abs(a[i-1]-v)<=abs(a[i]-v): return i-1
    return i

def _smooth(v,fb,r):
    s=pd.Series(v,dtype='float32').interpolate(limit_direction='both').fillna(fb)
    return (s.rolling(r*2+1,center=True,min_periods=1).mean() if r>0 else s).to_numpy(np.float32)

def beam_search(gr_h,tw_tvt,tw_gr,start_tvt,bs=10,mc=20.,es=144.,r=2):
    si=_nn(tw_tvt,start_tvt)
    sgr=_smooth(gr_h,float(np.nanmean(tw_gr)),r).astype(np.float64)
    return tw_tvt[_beam_jit(sgr,tw_gr.astype(np.float64),si,bs,float(mc),float(es))].astype(np.float32)

print("Warming up Numba JIT...")
_beam_jit(np.random.randn(30),np.random.randn(50),25,8,15.,100.)
print("Numba beam JIT ✓")


# === cell 4 ===
# ── Feature Engineering Helpers ───────────────────────────────────

def robust_slope(x,y):
    x=np.asarray(x,float); y=np.asarray(y,float)
    m=np.isfinite(x)&np.isfinite(y)
    if m.sum()<2 or np.std(x[m])<1e-6: return 0.
    return float(np.polyfit(x[m],y[m],1)[0])

def affine_cal(kgr,tw_at_k,min_pts=20):
    v=np.isfinite(kgr)&np.isfinite(tw_at_k)
    if v.sum()<min_pts or np.std(tw_at_k[v])<1e-6:
        return 1.,float(np.nanmean(kgr)-np.nanmean(tw_at_k)) if v.any() else 0.
    a,b=np.polyfit(tw_at_k[v],kgr[v],1)
    return float(a),float(b)

def wls_b_well(ktvt,kz,form_col,decay=0.02):
    """Recent-weighted b_well: tail points matter more."""
    n=len(ktvt)
    if n<3: return float(np.median(ktvt+kz-form_col))
    w=np.exp(decay*np.arange(n)); w/=w.sum()
    return float(np.dot(w,ktvt+kz-form_col))

def multi_scale_sc(kgr,ktvt,hgr,hws=(8,15,25),stride=3):
    """Multi-scale NCC: 3 window sizes → 3 independent TVT signals."""
    out=[]
    for hw in hws:
        win=2*hw+1; nk=len(kgr); nh=len(hgr)
        if nk<win+1 or nh==0:
            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        kg=pd.Series(kgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        hg=pd.Series(hgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        sts=np.arange(0,nk-win+1,stride,dtype=np.int32); M=len(sts)
        if M==0:
            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        C=kg[sts[:,None]+np.arange(win,dtype=np.int32)[None,:]].astype(np.float32)
        Cn=(C-C.mean(1,keepdims=True))/(C.std(1,keepdims=True)+1e-6)
        hp=np.pad(hg,hw,mode='edge')
        H=hp[np.arange(nh)[:,None]+np.arange(win)[None,:]].astype(np.float32)
        Hn=(H-H.mean(1,keepdims=True))/(H.std(1,keepdims=True)+1e-6)
        ncc=Hn@Cn.T/win
        best=ncc.argmax(1); score=ncc.max(1).astype(np.float32)
        ctrs=np.clip(sts[best]+hw,0,nk-1)
        out.append((ktvt[ctrs].astype(np.float32),score))
    return out

def gr_detrend_resid(gr_arr,md_arr):
    """Linear detrend GR → residual captures local anomalies."""
    m=np.isfinite(gr_arr)&np.isfinite(md_arr)
    if m.sum()<5: return gr_arr.copy()
    slope=robust_slope(md_arr[m],gr_arr[m])
    return (gr_arr-slope*md_arr).astype(np.float32)

print("Helpers OK ✓")


# === cell 5 ===
# ── Particle Filters (N=500, vectorised NumPy) ────────────────────

def _gr_sigma(hw,tw_tvt,tw_gr):
    kn=hw[hw['TVT_input'].notna()&hw['GR'].notna()]
    if len(kn)<20: return PF_GR_SIG_DEF
    return float(np.clip(np.std(kn['GR'].values-np.interp(kn['TVT_input'].values,tw_tvt,tw_gr)),
                          PF_GR_SIG_MIN,PF_GR_SIG_MAX))

def run_pf_ancc(hw,tw_tvt,tw_gr,N=ANCC_N):
    tmin,tmax=float(tw_tvt.min()),float(tw_tvt.max())
    gs=_gr_sigma(hw,tw_tvt,tw_gr)
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    tail=kn.tail(30); dt=np.diff(tail['TVT_input'].values)
    dz=np.diff(tail['Z'].values); dm=np.diff(tail['MD'].values); m=dm>0
    ir=float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.
    pos=(float(kn['TVT_input'].iloc[-1]+kn['Z'].iloc[-1])
         +np.random.normal(0.,ANCC_IS,N))
    rate=ir+np.random.normal(0.,ANCC_IR,N); w=np.ones(N)/N
    md_v=ev['MD'].values; z_v=ev['Z'].values; gr_v=ev['GR'].values
    pm=float(kn['MD'].iloc[-1])
    pts=np.empty(len(ev)); std_o=np.empty(len(ev))
    for i in range(len(ev)):
        dm2=max(md_v[i]-pm,1.)
        rate=ANCC_ALPHA*rate+np.random.normal(0.,ANCC_RN,N)
        pos +=rate*dm2+np.random.normal(0.,ANCC_PN,N)
        tvt_e=np.clip(pos-z_v[i],tmin-50.,tmax+50.); pos=tvt_e+z_v[i]
        if not np.isnan(gr_v[i]):
            eg=np.interp(tvt_e,tw_tvt,tw_gr)
            lk=np.exp(-0.5*((gr_v[i]-eg)/gs)**2); lk=np.maximum(lk,1e-300)
            w*=lk; ws=w.sum(); w=(w/ws) if ws>0 else np.full(N,1./N)
        ne=1./np.sum(w**2)
        if ne<PF_RESAMP*N:
            ix=np.searchsorted(np.cumsum(w),(np.arange(N)+np.random.uniform())/N)
            ix=np.clip(ix,0,N-1); pos=pos[ix]; rate=rate[ix]; w[:]=1./N
            pos+=np.random.normal(0.,ANCC_RP,N); rate+=np.random.normal(0.,ANCC_RR,N)
        tv=float(np.average(pos-z_v[i],weights=w)); pts[i]=tv
        std_o[i]=float(np.sqrt(np.average((pos-z_v[i]-tv)**2,weights=w)))
        pm=md_v[i]
    return pts.astype(np.float32),std_o.astype(np.float32)

def run_pf_z(hw,tw_tvt,tw_gr,N=PF_N):
    tw_s=pd.Series(tw_gr).rolling(PF_GR_WIN,center=True,min_periods=1).mean().values
    tf_p=interp1d(tw_tvt,tw_gr,bounds_error=False,fill_value=(tw_gr[0],tw_gr[-1]))
    tf_s=interp1d(tw_tvt,tw_s, bounds_error=False,fill_value=(tw_s[0], tw_s[-1]))
    tmin,tmax=tw_tvt.min(),tw_tvt.max()
    gs=_gr_sigma(hw,tw_tvt,tw_gr)
    kna=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    dz_k=np.diff(kna['Z'].values); dvt=np.diff(kna['TVT_input'].values)
    dmd_k=np.diff(kna['MD'].values); m2=dmd_k>0
    if m2.sum()>=10:
        vz=dz_k[m2]/dmd_k[m2]; vt=dvt[m2]/dmd_k[m2]
        A=np.column_stack([vz,np.ones_like(vz)])
        c,_,_,_=np.linalg.lstsq(A,vt,rcond=None)
        beta,icpt,zsig=c[0],c[1],max(np.std(vt-(c[0]*vz+c[1])),0.001)
    else: beta,icpt,zsig=-1.,0.,0.1
    tail2=kna.tail(20); dvt2=np.diff(tail2['TVT_input'].values)
    dmd2=np.diff(tail2['MD'].values); m3=dmd2>0
    iv=float(np.median(dvt2[m3]/dmd2[m3])) if m3.sum()>=3 else 0.
    gr_sm=hw['GR'].rolling(PF_GR_WIN,center=True,min_periods=1).mean()
    pos=float(kna['TVT_input'].iloc[-1])+np.random.normal(0.,PF_INIT_SPR,N)
    vel=iv+np.random.normal(0.,PF_INIT_V_STD,N); w=np.ones(N)/N
    md_v=ev['MD'].values; gr_v=ev['GR'].values; z_v=ev['Z'].values
    pm=float(kna['MD'].iloc[-1]); pz=float(kna['Z'].iloc[-1])
    pts=np.empty(len(ev)); std_o=np.empty(len(ev))
    for i,idx in enumerate(ev.index):
        dm=max(md_v[i]-pm,1.); dzd=(z_v[i]-pz)/dm; ve=beta*dzd+icpt
        vel=PF_MOM*vel+np.random.normal(0.,PF_VN,N)
        pos=pos+vel*dm+np.random.normal(0.,PF_PN,N)
        pos=np.clip(pos,tmin-50.,tmax+50.)
        if not np.isnan(gr_v[i]):
            ep=tf_p(pos); lp=np.exp(-0.5*((gr_v[i]-ep)/gs)**2)
            gsm=gr_sm.iloc[hw.index.get_loc(idx)]
            if not np.isnan(gsm):
                ls2=np.exp(-0.5*((gsm-tf_s(pos))/(gs*1.5))**2)
                lk=(1-PF_GR_WT)*lp+PF_GR_WT*ls2
            else: lk=lp
            lk=np.maximum(lk,1e-300); w*=lk; ws=w.sum()
            w=(w/ws) if ws>0 else np.full(N,1./N)
        lz=np.exp(-0.5*((vel-ve)/max(zsig*2.,0.005))**2)
        lz=np.maximum(lz,1e-300); w*=lz; ws=w.sum()
        w=(w/ws) if ws>0 else np.full(N,1./N)
        ne=1./np.sum(w**2)
        if ne<PF_RESAMP*N:
            ix=np.searchsorted(np.cumsum(w),(np.arange(N)+np.random.uniform())/N)
            ix=np.clip(ix,0,N-1); pos=pos[ix]; vel=vel[ix]; w[:]=1./N
            pos+=np.random.normal(0.,PF_ROUGH_P,N); vel+=np.random.normal(0.,PF_ROUGH_V,N)
        pts[i]=np.average(pos,weights=w)
        std_o[i]=np.sqrt(np.average((pos-pts[i])**2,weights=w))
        pm=md_v[i]; pz=z_v[i]
    return pts.astype(np.float32),std_o.astype(np.float32)

print("Particle Filters OK ✓")


# === cell 6 ===
# ── Spatial Imputers ───────────────────────────────────────────────

class FormationPlaneKNN:
    def __init__(self,well_ids,data_dir):
        rows=[]
        for wid in well_ids:
            p=data_dir/f'{wid}__horizontal_well.csv'
            try: df=pd.read_csv(p,usecols=['X','Y']+FORMATIONS).dropna()
            except: continue
            if len(df)==0: continue
            row={'wid':wid,'x':float(df['X'].median()),'y':float(df['Y'].median())}
            for c in FORMATIONS: row[f'{c}_m']=float(df[c].median())
            rows.append(row)
        self.df=pd.DataFrame(rows)
        self.wmap={w:i for i,w in enumerate(self.df['wid'])}
        xy=self.df[['x','y']].to_numpy()
        self.scale=np.where(xy.std(0)<1e-3,1.,xy.std(0))
        self.tree=cKDTree(xy/self.scale)
        self.xa=self.df['x'].to_numpy(); self.ya=self.df['y'].to_numpy()
        self.fa=self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)

    def impute(self,xy_q,self_wid=None,k=PLANE_K):
        q=xy_q/self.scale; nf=min(k+5,len(self.df))
        dist,idx=self.tree.query(q,k=nf,workers=-1)
        if self_wid in self.wmap: dist=np.where(idx==self.wmap[self_wid],np.inf,dist)
        ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
        dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
        vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.).astype(np.float64)
        xn=self.xa[ik]; yn=self.ya[ik]; fn=self.fa[ik]
        wx=w*xn; wy=w*yn
        A=np.zeros((len(q),3,3))
        A[:,0,0]=(wx*xn).sum(1); A[:,0,1]=(wx*yn).sum(1); A[:,0,2]=wx.sum(1)
        A[:,1,0]=A[:,0,1]; A[:,1,1]=(wy*yn).sum(1); A[:,1,2]=wy.sum(1)
        A[:,2,0]=A[:,0,2]; A[:,2,1]=A[:,1,2]; A[:,2,2]=w.sum(1)
        A[:,0,0]+=1e-9; A[:,1,1]+=1e-9; A[:,2,2]+=1e-9
        rhs=np.stack([(wx[:,:,None]*fn).sum(1),(wy[:,:,None]*fn).sum(1),(w[:,:,None]*fn).sum(1)],1)
        try: coef=np.linalg.solve(A,rhs)
        except:
            coef=np.zeros((len(q),3,6))
            for r in range(len(q)):
                try: coef[r]=np.linalg.pinv(A[r])@rhs[r]
                except: pass
        Xq=xy_q[:,0]; Yq=xy_q[:,1]
        pred=(Xq[:,None]*coef[:,0,:]+Yq[:,None]*coef[:,1,:]+coef[:,2,:]).astype(np.float32)
        pred[~vk.any(1)]=self.fa.mean(0)
        return pred,np.where(vk,dk,np.inf).min(1).astype(np.float32)

class DenseANCCImputer:
    def __init__(self,well_ids,data_dir,spw=DENSE_SPW):
        xs,ys,anccs,wids=[],[],[],[]
        for wid in well_ids:
            p=data_dir/f'{wid}__horizontal_well.csv'
            try: df=pd.read_csv(p,usecols=['X','Y','ANCC']).dropna()
            except: continue
            if len(df)==0: continue
            ix=np.linspace(0,len(df)-1,min(spw,len(df)),dtype=int)
            s=df.iloc[ix]
            xs.append(s['X'].values); ys.append(s['Y'].values)
            anccs.append(s['ANCC'].values); wids.extend([wid]*len(s))
        self.xy=np.column_stack([np.concatenate(xs),np.concatenate(ys)])
        self.ancc=np.concatenate(anccs).astype(np.float32)
        self.wids=np.array(wids)
        self.scale=np.where(self.xy.std(0)<1e-3,1.,self.xy.std(0))
        self.tree=cKDTree(self.xy/self.scale)

    def impute(self,xy_q,self_wid=None,k=DENSE_K,nfetch=3000):
        xy_q=np.atleast_2d(xy_q); q=xy_q/self.scale
        nf=min(nfetch,len(self.ancc))
        dist,idx=self.tree.query(q,k=nf,workers=-1)
        if self_wid: dist=np.where(self.wids[idx]==self_wid,np.inf,dist)
        ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
        dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
        vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.)
        sw=w.sum(1); safe=np.where(sw<1e-9,1.,sw)
        an=self.ancc[ik]
        ap=(an*w).sum(1)/safe; ap=np.where(sw<1e-9,float(self.ancc.mean()),ap)
        var=((an-ap[:,None])**2*w).sum(1)/safe
        return ap.astype(np.float32),np.sqrt(np.maximum(var,0.)).astype(np.float32),np.where(vk,dk,np.inf).min(1).astype(np.float32)

hw_paths=sorted(TRAIN_DIR.glob('*__horizontal_well.csv'))
train_wids=[p.stem.replace('__horizontal_well','') for p in hw_paths]
print(f"Building imputers ({len(train_wids)} wells)..."); t0=time.time()
FI=FormationSurfaceV2(train_wids, method='gp_per_surface')
DI=DenseANCCImputer(train_wids,TRAIN_DIR)
print(f"  FPK: {len(FI.df)} centroids | Dense: {len(DI.ancc):,} pts  ({time.time()-t0:.0f}s)")


# === cell 7 ===
# ── Per-Well Feature Builder ───────────────────────────────────────
_FI=FI; _DI=DI

ANCH_OFFS=np.array([-80,-40,-20,-10,-5,0,5,10,20,40,80],np.float32)
BEAM_OFFS=np.array([-40,-20,-10,-5,-3,0,3,5,10,20,40],np.float32)
SC_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)
PF_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)  # 4th family

def build_well(hw_path,tw_path,is_train):
    global _FI,_DI
    wid=Path(hw_path).stem.replace('__horizontal_well','')
    try:
        hw=pd.read_csv(hw_path); tw=pd.read_csv(tw_path).sort_values('TVT')
    except: return None
    if is_train and 'TVT' not in hw.columns: return None
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0 or len(kn)<10: return None
    if is_train and hw['TVT'].isna().all(): return None

    tw_tvt=tw['TVT'].to_numpy(np.float32); tw_gr=tw['GR'].to_numpy(np.float32)
    if len(tw_tvt)<3: return None

    # ── Particle Filters ────────────────────────────────────────
    pf_a,std_a=run_pf_ancc(hw,tw_tvt,tw_gr)
    if len(pf_a)==0: return None
    pf_z,std_z=run_pf_z(hw,tw_tvt,tw_gr)
    pf_use=pf_a; std_use=std_a
    has_z=(len(pf_z)==len(pf_a) and not np.any(np.isnan(pf_z)))

    # ── GR ──────────────────────────────────────────────────────
    lk=kn.iloc[-1]; last_tvt=float(lk['TVT_input'])
    gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    hgr=gr_full.iloc[ev.index[0]:].to_numpy(np.float32)
    kgr=gr_full.iloc[:len(kn)].to_numpy(np.float32)
    hmd_s=ev['MD'].to_numpy(np.float32)

    # GR detrend residual (new)
    gr_arr=gr_full.values.astype(np.float32)
    md_arr=hw['MD'].values.astype(np.float32)
    gr_detr=gr_detrend_resid(gr_arr,md_arr)
    hgr_detr=gr_detr[ev.index]

    # ── 7 Beam configs ──────────────────────────────────────────
    bpaths={}
    for (bs,mc,es,r,tag) in BEAMS:
        bpaths[tag]=beam_search(hgr,tw_tvt,tw_gr,last_tvt,bs,mc,es,r)
    beam_ref=(bpaths['cons']+bpaths['sm5'])/2.

    # ── Multi-scale NCC self-corr ────────────────────────────────
    ktvt=kn['TVT_input'].to_numpy(np.float32)
    sc_res=multi_scale_sc(kgr,ktvt,hgr,hws=(8,15,25),stride=3)
    sc8,sc8s=sc_res[0]; sc15,sc15s=sc_res[1]; sc25,sc25s=sc_res[2]
    sc_cons=(sc8+sc15+sc25)/3.
    sc_trust=float(np.clip(len(kn)/200.,0.,0.6))
    hyb_ref=(1-sc_trust)*beam_ref+sc_trust*sc15

    # ── Affine calibration ───────────────────────────────────────
    tw_at_k=np.interp(ktvt,tw_tvt,tw_gr).astype(np.float32)
    a_cal,b_cal=affine_cal(kgr,tw_at_k)

    # ── Prefix stats ─────────────────────────────────────────────
    kmd=kn['MD'].to_numpy(np.float32); kz=kn['Z'].to_numpy(np.float32)
    pfx_rmse=float(np.sqrt(np.mean((kgr-tw_at_k)**2)))
    slp_all=robust_slope(kmd,ktvt); slp_50=robust_slope(kmd[-50:],ktvt[-50:])
    slp_z=robust_slope(kz,ktvt)
    # Prefix GR slope (trend of GR in known zone)
    pfx_gr_slope=robust_slope(kmd,kgr)

    # ── Spatial imputation ───────────────────────────────────────
    swid=wid if is_train else None
    xy_ev=ev[['X','Y']].to_numpy(np.float64)
    xy_kn=kn[['X','Y']].to_numpy(np.float64)
    form_ev,knn_d=_FI.impute(xy_ev,self_wid=swid)
    form_kn,_   =_FI.impute(xy_kn,self_wid=swid)

    z_kn=kn['Z'].to_numpy(np.float32); z_ev=ev['Z'].to_numpy(np.float32)

    # Per-formation TVT + WLS b_well + known-zone RMSE
    tvt_fs={}; form_rmse={}; form_list=[]
    for fi2,fn in enumerate(FORMATIONS):
        b_v=ktvt+z_kn-form_kn[:,fi2]
        b_all=float(np.median(b_v))
        b_wls=wls_b_well(ktvt,z_kn,form_kn[:,fi2])
        b_50 =float(np.median(b_v[-50:])) if len(b_v)>=5 else b_all
        tvt_f =(-z_ev+form_ev[:,fi2]+b_all).astype(np.float32)
        tvt_fw=(-z_ev+form_ev[:,fi2]+b_wls).astype(np.float32)
        tvt_fs[fn]=tvt_f; tvt_fs[fn+'_wls']=tvt_fw
        tvt_fs[f'bw_{fn}']=np.float32(b_all)
        tvt_fs[f'bw50_{fn}']=np.float32(b_50)
        tvt_fs[f'bww_{fn}']=np.float32(b_wls)
        # Known-zone RMSE for this formation
        form_rmse[fn]=float(np.sqrt(np.mean((ktvt-(-z_kn+form_kn[:,fi2]+b_all))**2)))
        form_list.append(tvt_f)

    # Formation consensus features
    fs=np.stack(form_list,1)  # (nh,6)
    form_mean_d=(fs.mean(1)-last_tvt).astype(np.float32)
    form_std_d =fs.std(1).astype(np.float32)
    form_rng_d =(fs.max(1)-fs.min(1)).astype(np.float32)

    # Dense ANCC + WLS
    d_ancc,d_std,d_dist=_DI.impute(xy_ev,self_wid=swid)
    d_kn,d_std_kn,_=_DI.impute(xy_kn,self_wid=swid)
    b_vd=ktvt+z_kn-d_kn
    b_d=float(np.median(b_vd)); b_d_wls=wls_b_well(ktvt,z_kn,d_kn)
    b_d50=float(np.median(b_vd[-50:])) if len(b_vd)>=5 else b_d
    tvt_dense =(-z_ev+d_ancc+b_d  ).astype(np.float32)
    tvt_densew=(-z_ev+d_ancc+b_d_wls).astype(np.float32)
    tvt_d50   =(-z_ev+d_ancc+b_d50).astype(np.float32)
    res_kn=ktvt+z_kn-d_kn
    d_rmse=float(np.sqrt(np.mean(res_kn**2))); d_bias=float(np.mean(res_kn))

    # Inter-signal consensus (all TVT estimates)
    all_sigs=[pf_use,*(p for p in bpaths.values()),sc8,sc15,sc25,
               tvt_fs['ANCC'],tvt_dense]
    sig_mat=np.stack(all_sigs,1)
    sig_std=sig_mat.std(1).astype(np.float32)
    sig_mean=(sig_mat.mean(1)-last_tvt).astype(np.float32)

    # GR rolling features
    gr_s=pd.Series(gr_full.values)
    rolls={}
    for w in [5,21,51,101]:
        r=gr_s.rolling(w,center=True,min_periods=1)
        rolls[f'grm{w}']=r.mean().iloc[ev.index].values.astype(np.float32)
        rolls[f'grs{w}']=r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
    for lag in [1,5,15,30]:
        rolls[f'glag{lag}']=gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32)
        rolls[f'glead{lag}']=gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
    gr_d1=gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_d2=gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    # GR envelope (rolling max) + energy (rolling RMS)
    gr_env=gr_s.rolling(21,center=True,min_periods=1).max().iloc[ev.index].values.astype(np.float32)
    gr_nrg=np.sqrt(np.maximum((gr_s**2).rolling(21,center=True,min_periods=1).mean(),0.)
                   ).iloc[ev.index].values.astype(np.float32)

    # Slope baselines
    md_since=hmd_s-float(lk['MD'])
    slp_b_all=(last_tvt+slp_all*md_since).astype(np.float32)
    slp_b_50 =(last_tvt+slp_50 *md_since).astype(np.float32)

    # Trajectory
    mdd=hw['MD'].diff().replace(0,np.nan)
    dzdmd=(hw['Z'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
    dxdmd=(hw['X'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
    dydmd=(hw['Y'].diff()/mdd).iloc[ev.index].values.astype(np.float32)

    nh=len(ev); frac=(np.arange(nh)/max(nh-1,1)).astype(np.float32)
    def sc(v): return np.full(nh,np.float32(v),np.float32)

    feats={
        'well':wid,'id':[f'{wid}_{i}' for i in ev.index],
        'last_known_tvt':sc(last_tvt),
        # PF
        'pf_ancc':pf_use,'pf_ancc_std':std_use,
        'pf_ancc_d':(pf_use-last_tvt).astype(np.float32),
        'pf_z':(pf_z.astype(np.float32) if has_z else sc(last_tvt)),
        'pf_z_d':((pf_z-last_tvt).astype(np.float32) if has_z else sc(0.)),
        'pf_vs_z':((pf_use-pf_z.astype(np.float32)) if has_z else sc(0.)),
        # Beam (7)
        **{f'beam_{t}_d':(p-np.float32(last_tvt)).astype(np.float32) for t,p in bpaths.items()},
        'beam_mean_d':np.stack([(p-last_tvt) for p in bpaths.values()],1).mean(1).astype(np.float32),
        'beam_std_d': np.stack([(p-last_tvt) for p in bpaths.values()],1).std(1).astype(np.float32),
        'beam_med_d': np.median(np.stack([(p-last_tvt) for p in bpaths.values()],1),1).astype(np.float32),
        # Multi-scale NCC
        'sc8_d' :(sc8 -np.float32(last_tvt)).astype(np.float32),'sc8_score' :sc8s,
        'sc15_d':(sc15-np.float32(last_tvt)).astype(np.float32),'sc15_score':sc15s,
        'sc25_d':(sc25-np.float32(last_tvt)).astype(np.float32),'sc25_score':sc25s,
        'sc_cons_d':(sc_cons-np.float32(last_tvt)).astype(np.float32),
        'sc_trust':sc(sc_trust),'hyb_d':(hyb_ref-np.float32(last_tvt)).astype(np.float32),
        # Inter-signal consensus
        'signal_std':sig_std,'signal_mean_d':sig_mean,
        # Formation TVT (median + WLS b_well, all 6)
        **{f'tvtF_{fn}_d':(tvt_fs[fn]-last_tvt).astype(np.float32)      for fn in FORMATIONS},
        **{f'tvtFw_{fn}_d':(tvt_fs[fn+'_wls']-last_tvt).astype(np.float32) for fn in FORMATIONS},
        **{f'bw_{fn}':tvt_fs[f'bw_{fn}']     for fn in FORMATIONS},
        **{f'bww_{fn}':tvt_fs[f'bww_{fn}']   for fn in FORMATIONS},
        **{f'frm_rmse_{fn}':sc(form_rmse[fn]) for fn in FORMATIONS},
        'form_mean_d':form_mean_d,'form_std_d':form_std_d,'form_rng_d':form_rng_d,
        'knn_d':knn_d,
        # Dense ANCC
        'dense_ancc':d_ancc,'dense_std':d_std,'dense_dist':d_dist,
        'tvt_dense_d' :(tvt_dense -last_tvt).astype(np.float32),
        'tvt_densew_d':(tvt_densew-last_tvt).astype(np.float32),
        'tvt_d50_d'   :(tvt_d50   -last_tvt).astype(np.float32),
        'dense_rmse':sc(d_rmse),'dense_bias':sc(d_bias),
        # Cross-signal
        'pf_vs_form':(pf_use-tvt_fs['ANCC']).astype(np.float32),
        'pf_vs_dense':(pf_use-tvt_dense).astype(np.float32),
        'form_vs_dense':(tvt_fs['ANCC']-tvt_dense).astype(np.float32),
        'beam_vs_form':(bpaths['cons']-tvt_fs['ANCC']).astype(np.float32),
        'sc_vs_beam':(sc15-bpaths['cons']).astype(np.float32),
        # Affine
        'cal_a':sc(a_cal),'cal_b':sc(b_cal),
        # Prefix
        'pfx_rmse':sc(pfx_rmse),'known_len':sc(len(kn)),'eval_len':sc(nh),
        'slp_all':sc(slp_all),'slp_50':sc(slp_50),'slp_z':sc(slp_z),
        'pfx_gr_slope':sc(pfx_gr_slope),   # NEW
        'slp_b_d_all':(slp_b_all-last_tvt).astype(np.float32),
        'slp_b_d_50' :(slp_b_50 -last_tvt).astype(np.float32),
        'ktvt_range':sc(float(np.ptp(ktvt))),'ktvt_std':sc(float(ktvt.std())),
        # Position
        'md_since':md_since,'frac':frac,'frac2':frac**2,'sqrt_frac':np.sqrt(frac),
        'z':z_ev,
        'dx':(ev['X']-float(lk['X'])).to_numpy(np.float32),
        'dy':(ev['Y']-float(lk['Y'])).to_numpy(np.float32),
        'dz':(z_ev-float(lk['Z'])).astype(np.float32),
        'dxy':np.sqrt((ev['X']-float(lk['X']))**2+(ev['Y']-float(lk['Y']))**2).to_numpy(np.float32),
        'dzdmd':dzdmd,'dxdmd':dxdmd,'dydmd':dydmd,
        # GR
        'gr':hgr,'gr_d1':gr_d1,'gr_d2':gr_d2,
        'gr_env':gr_env,'gr_nrg':gr_nrg,
        'gr_detr':hgr_detr,                # NEW: detrended GR
        'gr_vs_tw':hgr-np.float32(np.interp(last_tvt,tw_tvt,tw_gr)),
        'gr_vs_slp':hgr-np.interp(slp_b_all,tw_tvt,tw_gr).astype(np.float32),
        # tw_diff: 4 families (anchor, beam, sc, PF)
        **{f'tda{int(o)}' :hgr-np.float32(np.interp(last_tvt+o,tw_tvt,tw_gr))         for o in ANCH_OFFS},
        **{f'tdbc{int(o)}':hgr-np.interp(beam_ref+o,tw_tvt,tw_gr).astype(np.float32)  for o in BEAM_OFFS},
        **{f'tdsc{int(o)}':hgr-np.interp(sc15+o,tw_tvt,tw_gr).astype(np.float32)      for o in SC_OFFS},
        **{f'tdpf{int(o)}':hgr-np.interp(pf_use+o,tw_tvt,tw_gr).astype(np.float32)    for o in PF_OFFS},  # NEW 4th family
        'tw_range':sc(float(np.ptp(tw_tvt))),'tw_gr_mean':sc(float(tw_gr.mean())),
    }
    for k,v in rolls.items(): feats[k]=v

    result=pd.DataFrame(feats)
    if is_train:
        if 'TVT' not in ev.columns or ev['TVT'].isna().all(): return None
        result['target']=(ev['TVT'].to_numpy(np.float32)-np.float32(last_tvt))
    return result

def build_dataset(paths,is_train,label):
    args=[(str(p),str(p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv'),is_train)
          for p in paths
          if (p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv').exists()]
    print(f"  {label}: {len(args)} wells | {NCPU} threads")
    res=Parallel(n_jobs=NCPU,prefer='threads',verbose=3)(
        delayed(build_well)(hp,tp,it) for hp,tp,it in args)
    parts=[r for r in res if r is not None]
    print(f"  {label}: OK={len(parts)} skipped={len(args)-len(parts)}")
    return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()

print("Feature builder OK ✓")


# === cell 8 ===
print("Building train..."); t0=time.time()
train_df=build_dataset(hw_paths,is_train=True,label="train")
print(f"  train: {train_df.shape}  ({time.time()-t0:.0f}s)")

test_paths=sorted(TEST_DIR.glob('*__horizontal_well.csv'))
print("Building test..."); t0=time.time()
test_df=build_dataset(test_paths,is_train=False,label="test")
print(f"  test:  {test_df.shape}  ({time.time()-t0:.0f}s)")

SKIP={'well','id','target'}
feature_cols=[c for c in train_df.columns if c not in SKIP]
print(f"#features: {len(feature_cols)}")

X=train_df[feature_cols]; y=train_df['target']; g=train_df['well']
Xt=test_df[feature_cols]
gc.collect()


# === cell 9 ===
# ── Train: LightGBM×3 diverse configs + CatBoost ─────────────────
cv=GroupKFold(n_splits=N_SPLITS); splits=list(cv.split(X,y,g))

def run_lgb(cfg_idx):
    cfg=LGB_CONFIGS[cfg_idx]; p=dict(LGB_BASE,**cfg)
    n_est=p.pop('n_estimators')
    oof=np.zeros(len(train_df),np.float32); tp=np.zeros(len(test_df),np.float32)
    for fold,(tr,va) in enumerate(splits):
        dtr=lgb.Dataset(X.iloc[tr],label=y.iloc[tr])
        dva=lgb.Dataset(X.iloc[va],label=y.iloc[va],reference=dtr)
        m=lgb.train(p,dtr,valid_sets=[dva],num_boost_round=n_est,
                    callbacks=[lgb.early_stopping(250,verbose=False),lgb.log_evaluation(800)])
        oof[va]=m.predict(X.iloc[va],num_iteration=m.best_iteration).astype(np.float32)
        tp+=m.predict(Xt,num_iteration=m.best_iteration).astype(np.float32)/N_SPLITS
        print(f"  LGB{cfg_idx} f{fold}: rmse={root_mean_squared_error(y.iloc[va],oof[va]):.4f}"
              f"  iter={m.best_iteration}")
    r=root_mean_squared_error(y,oof)
    print(f"  LGB{cfg_idx} OOF={r:.4f}"); return oof,tp,r

def run_cb():
    oof=np.zeros(len(train_df),np.float32); tp=np.zeros(len(test_df),np.float32)
    for fold,(tr,va) in enumerate(splits):
        m=CatBoostRegressor(**CB_PARAMS)
        m.fit(Pool(X.iloc[tr].values,label=y.iloc[tr].values),
              eval_set=Pool(X.iloc[va].values,label=y.iloc[va].values),
              use_best_model=True)
        oof[va]=m.predict(X.iloc[va].values).astype(np.float32)
        tp+=m.predict(Xt.values).astype(np.float32)/N_SPLITS
        print(f"  CB f{fold}: rmse={root_mean_squared_error(y.iloc[va],oof[va]):.4f}")
    r=root_mean_squared_error(y,oof)
    print(f"  CB OOF={r:.4f}"); return oof,tp,r

def run_xgb(seed=42):
    p=dict(XGB_PARAMS); p['random_state']=seed
    oof=np.zeros(len(train_df),np.float32); tp=np.zeros(len(test_df),np.float32)
    for fold,(tr,va) in enumerate(splits):
        m=xgb.XGBRegressor(**p)
        m.fit(X.iloc[tr],y.iloc[tr],eval_set=[(X.iloc[va],y.iloc[va])],verbose=False)
        oof[va]=m.predict(X.iloc[va]).astype(np.float32)
        tp+=m.predict(Xt).astype(np.float32)/N_SPLITS
        print(f"  XGB f{fold}: rmse={root_mean_squared_error(y.iloc[va],oof[va]):.4f}"
              f"  iter={m.best_iteration}")
    r=root_mean_squared_error(y,oof)
    print(f"  XGB OOF={r:.4f}"); return oof,tp,r

results={}
for i in range(3):
    oof,tp,r=run_lgb(i); results[f'lgb{i}']={'oof':oof,'test':tp,'rmse':r}
oof,tp,r=run_cb(); results['cb']={'oof':oof,'test':tp,'rmse':r}
oof,tp,r=run_xgb(); results['xgb']={'oof':oof,'test':tp,'rmse':r}

# v2: persist per-model OOF + test preds for offline blend exploration
if not _kaggle_working:
    artifact_dir = Path(os.environ.get("ROGII_ARTIFACT_DIR", "./experiments/v2_artifacts"))
    artifact_dir.mkdir(parents=True, exist_ok=True)
    for k, v in results.items():
        np.save(artifact_dir / f"{k}_oof.npy", v['oof'])
        np.save(artifact_dir / f"{k}_test.npy", v['test'])
    np.save(artifact_dir / "y_residual.npy", y.values)
    np.save(artifact_dir / "groups.npy", g)
    train_df.to_parquet(artifact_dir / "train_df.parquet")
    test_df.to_parquet(artifact_dir / "test_df.parquet")
    print(f"Saved per-model OOF artifacts to {artifact_dir}")

# v2: Compare three ensemble strategies — simple avg, Ridge(positive), Nelder-Mead
keys=list(results.keys())
Sx=np.column_stack([results[k]['oof'] for k in keys])
St=np.column_stack([results[k]['test'] for k in keys])

# 1) Simple average
r_avg=root_mean_squared_error(y,Sx.mean(1))

# 2) Ridge positive
ridge=Ridge(alpha=1.,fit_intercept=False,positive=True)
ridge.fit(Sx,y.values)
oof_ridge=ridge.predict(Sx); test_ridge=ridge.predict(St)
r_ridge=root_mean_squared_error(y,oof_ridge)
ridge_wts=ridge.coef_/max(ridge.coef_.sum(),1e-9)

# 3) Nelder-Mead with 10 random restarts (Karnakbaev style)
def _nm_loss(w_logits, Sx_=Sx, y_=y.values):
    w=np.exp(w_logits); w=w/w.sum()
    return float(np.sqrt(((y_-Sx_@w)**2).mean()))

rng=np.random.default_rng(42)
best_w_nm,best_r_nm=None,np.inf
for restart in range(10):
    w0=rng.normal(scale=0.5,size=Sx.shape[1])
    opt=minimize(_nm_loss,w0,method='Nelder-Mead',
                 options={'xatol':1e-6,'fatol':1e-6,'maxiter':10000})
    if opt.fun<best_r_nm:
        w=np.exp(opt.x); w=w/w.sum()
        best_w_nm,best_r_nm=w,opt.fun
oof_nm=Sx@best_w_nm; test_nm=St@best_w_nm
r_nm=root_mean_squared_error(y,oof_nm)

print(f"\n=== Ensemble strategies (residual RMSE) ===")
print(f"  Simple avg:    {r_avg:.4f}")
print(f"  Ridge weights: {dict(zip(keys,ridge_wts.round(4)))}")
print(f"  Ridge stack:   {r_ridge:.4f}")
print(f"  NM weights:    {dict(zip(keys,best_w_nm.round(4)))}")
print(f"  NM blend:      {r_nm:.4f}")

# Pick best on OOF
strategies=[
    ('simple_avg',Sx.mean(1),St.mean(1),r_avg),
    ('ridge',oof_ridge,test_ridge,r_ridge),
    ('nm',oof_nm,test_nm,r_nm),
]
best=min(strategies,key=lambda x:x[3])
print(f"\nBEST strategy: {best[0]}  RMSE={best[3]:.4f}")
final_oof,final_test=best[1],best[2]


# === cell 10 ===
# ── Post-Processing + Submission ──────────────────────────────────
base=train_df['last_known_tvt'].values; ytrue=y.values+base

# Grid search: blend weight with PF signal + alpha + fade-in
# New: also try blending final prediction with raw PF ANCC at weight w_pf
print("Grid search post-processing...")
best_cfg,best_r=(None,None,None,None),np.inf
pf_oof=(train_df['pf_ancc'].values-base)  # PF residual

for alpha in np.arange(0.65,1.01,0.05):
    for tau in [None,25.,50.,100.,200.]:
        for w_pf in [0.0,0.05,0.10]:        # blend with PF signal
            d=final_oof*(1-w_pf)+pf_oof*w_pf
            if tau: d*=(1.-np.exp(-np.maximum(train_df['md_since'].values,0.)/tau))
            d*=alpha
            r=root_mean_squared_error(ytrue,base+d)
            if r<best_r: best_r,best_cfg=r,(alpha,tau,w_pf,None)
print(f"Best: alpha={best_cfg[0]:.2f} tau={best_cfg[1]} w_pf={best_cfg[2]:.2f}  abs TVT RMSE={best_r:.4f}")
ALPHA,TAU,W_PF=best_cfg[0],best_cfg[1],best_cfg[2]

def apply_pp(df,delta,pf_delta,alpha,tau,w_pf):
    d=delta*(1-w_pf)+pf_delta*w_pf
    if tau: d*=(1.-np.exp(-np.maximum(df['md_since'].values,0.)/tau))
    return d*alpha

def sg_smooth(df,col,sg_w=17,sg_p=3):
    df=df.copy()
    for well,g in df.groupby('well',sort=False):
        v=g[col].values; n=len(v); wl=min(sg_w,n)
        if wl%2==0: wl-=1
        if wl>=sg_p+2: v=savgol_filter(v,wl,sg_p)
        df.loc[g.index,col]=v
    return df

test_df2=test_df.copy()
pf_test=(test_df2['pf_ancc'].values-test_df2['last_known_tvt'].values)
test_df2['pred']=(test_df2['last_known_tvt'].values
                 +apply_pp(test_df2,final_test,pf_test,ALPHA,TAU,W_PF))
test_df2=sg_smooth(test_df2,'pred')

sample=pd.read_csv(SAMPLE)
sub=(sample[['id']].merge(
     test_df2[['id','pred']].rename(columns={'pred':'tvt'}),on='id',how='left'))
fb=float(train_df['last_known_tvt'].mean()+train_df['target'].mean())
sub['tvt']=sub['tvt'].fillna(fb)
sub[['id','tvt']].to_csv(OUT,index=False)

print(f"\n✅  {OUT}  {len(sub)} rows")
print("\n─── Final Summary ─────────────────────────")
for k,v in results.items(): print(f"  {k:8s}: OOF residual RMSE = {v['rmse']:.4f}")
r_best=min(r_avg,r_ridge,r_nm)
print(f"  {'best_blend':10s}: OOF residual RMSE = {r_best:.4f}")
print(f"  {'PostProc':8s}: OOF absolute  RMSE = {best_r:.4f}")
print(sub.head(8).to_string(index=False))

